# Phase 6 · Notebook 02 — Ensemble Evaluation

The single biggest reliable lift in NER is usually an ensemble of
independent predictors. The ensemble's job is to absorb the strengths
of each member while averaging out their idiosyncratic mistakes.

We combine:

- **spaCy `en_core_web_trf`** — strong general-purpose English NER (Phase 1 anchor).
- **LegalBERT fine-tuned on TAB train** — produced by Notebook 01.
- **Microsoft Presidio** — NER + the custom CASE_NUMBER regex from Phase 2.

These three are *complementary* by design: spaCy is broad and unbiased,
LegalBERT is legal-domain-specific, Presidio's regex layer catches the
structured identifiers neither NER catches reliably.

Voting policy: keep any span supported by ≥1 predictor (union — recall-
maximising). Drop to `min_votes=2` later if precision becomes a problem.

---


In [1]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate spacy
# !python -m spacy download en_core_web_lg


## Setup


In [2]:
import sys
sys.path.insert(0, "../src")

import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.mapping import SPACY_TO_TAB
from anonymisation.evaluation import evaluate_document, merge_results, results_to_dataframe
from anonymisation.ensemble import EnsemblePredictor
from anonymisation.predictors import (
    make_hf_predictor, make_presidio_predictor, make_finetuned_predictor,
    build_presidio_analyzer,
)
from anonymisation.device import best_device


## Build the three predictors


In [3]:
# --- spaCy ---
import spacy
SPACY_MODEL = "en_core_web_trf"
print(f"Loading spaCy: {SPACY_MODEL}")
nlp = spacy.load(SPACY_MODEL)

def spacy_predict(text):
    return [
        (e.start_char, e.end_char, SPACY_TO_TAB[e.label_], e.text)
        for e in nlp(text).ents if e.label_ in SPACY_TO_TAB
    ]


Loading spaCy: en_core_web_trf


In [4]:
# --- LegalBERT fine-tuned (from Notebook 01) ---
LEGALBERT_DIR = "../models/legalbert-tab/final"

from transformers import AutoModelForTokenClassification, AutoTokenizer
device, _ = best_device()
print(f"Loading LegalBERT from {LEGALBERT_DIR} on {device}")
lb_tok = AutoTokenizer.from_pretrained(LEGALBERT_DIR)
lb_model = AutoModelForTokenClassification.from_pretrained(LEGALBERT_DIR)
legalbert_predict = make_finetuned_predictor(lb_model, lb_tok, device=device)


Loading LegalBERT from checkpoints/legalbert-tab/final on mps


In [5]:
# --- Presidio with the custom CASE_NUMBER recogniser ---
print("Building Presidio analyzer (en_core_web_lg backbone)")
analyzer = build_presidio_analyzer(add_case_number_recognizer=True, spacy_model="en_core_web_lg")
presidio_predict = make_presidio_predictor(analyzer)


Building Presidio analyzer (en_core_web_lg backbone)


## Wire the ensemble


In [6]:
ensemble = EnsemblePredictor(
    predictors={
        "spacy":     spacy_predict,
        "legalbert": legalbert_predict,
        "presidio":  presidio_predict,
    },
    min_votes=1,         # union — keep every span supported by ≥1 predictor
    tie_break="longest", # on overlapping spans, keep the longest
)

# Smoke test on a synthetic sentence
demo = "Maria Petrova (Application no. 12345/67) sued Acme Holdings Ltd in Sofia District Court."
print("Ensemble output:")
for s, e, t, txt in ensemble(demo):
    print(f"  [{t:8s}] {txt!r}  ({s}:{e})")


Ensemble output:
  [PERSON  ] 'Maria Petrova'  (0:13)
  [CODE    ] 'Application no. 12345/67'  (15:39)
  [ORG     ] 'Acme Holdings Ltd'  (46:63)
  [ORG     ] 'Sofia District Court'  (67:87)


## Evaluate on TAB test


In [7]:
dataset = load_tab()
test_docs = list(dataset["test"])
RESULTS_PATH = "../results/null_ensemble.csv"

all_merged = {}
for mode in ["partial", "exact"]:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}  ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(ensemble, doc, mode=mode))
    print(f"  done in {time.time() - start:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "ensemble_v1")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")



--- partial match ---
  50/555  (1.7 docs/s)
  100/555  (1.7 docs/s)
  150/555  (1.7 docs/s)
  200/555  (1.7 docs/s)
  250/555  (1.8 docs/s)
  300/555  (1.9 docs/s)
  350/555  (2.0 docs/s)
  400/555  (2.1 docs/s)
  450/555  (2.3 docs/s)
  500/555  (2.1 docs/s)
  550/555  (2.1 docs/s)
  done in 266.8s

--- exact match ---
  50/555  (1.8 docs/s)
  100/555  (1.7 docs/s)
  150/555  (1.7 docs/s)
  200/555  (1.7 docs/s)
  250/555  (1.8 docs/s)
  300/555  (1.9 docs/s)
  350/555  (2.0 docs/s)
  400/555  (2.1 docs/s)
  450/555  (2.2 docs/s)
  500/555  (2.1 docs/s)
  550/555  (2.1 docs/s)
  done in 270.7s

Saved → ../results/ensemble_results.csv


## Per-entity results


In [8]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in ["partial", "exact"]:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))



── PARTIAL MATCH ──
   Entity    TP    FP   FN Precision Recall    F1
   PERSON  3977   975  161     80.3%  96.1% 87.5%
      ORG  1644  9754  302     14.4%  84.5% 24.6%
      LOC  1391  2955   68     32.0%  95.3% 47.9%
 DATETIME  9433  2120  103     81.6%  98.9% 89.5%
 QUANTITY   647  4774   17     11.9%  97.4% 21.3%
     CODE   348    28 1264     92.6%  21.6% 35.0%
      DEM   423   540  494     43.9%  46.1% 45.0%
     MISC    59  4940  478      1.2%  11.0%  2.1%
▶ OVERALL 17922 26086 2887     40.7%  86.1% 55.3%

── EXACT MATCH ──
   Entity    TP    FP   FN Precision Recall    F1
   PERSON  3678  1274  460     74.3%  88.9% 80.9%
      ORG   758 10640 1188      6.7%  39.0% 11.4%
      LOC  1213  3133  246     27.9%  83.1% 41.8%
 DATETIME  8842  2711  694     76.5%  92.7% 83.9%
 QUANTITY   431  4990  233      8.0%  64.9% 14.2%
     CODE   268   108 1344     71.3%  16.6% 27.0%
      DEM   333   630  584     34.6%  36.3% 35.4%
     MISC    12  4987  525      0.2%   2.2%  0.4%
▶ OVERALL 

## Try the alternative voting policies

`min_votes=1` is the union (most recall). `min_votes=2` is the
intersection-like middle. `min_votes=3` is strict consensus (highest precision,
lowest recall). Run a quick sample to see the precision/recall trade-off.


In [9]:
sample = test_docs[:50]
for k in (1, 2, 3):
    ensemble.min_votes = k
    per_doc = [evaluate_document(ensemble, d, mode="partial") for d in sample]
    merged = merge_results(per_doc)["_ALL"]
    print(f"  min_votes={k}:  P={merged.precision:.1%}  R={merged.recall:.1%}  F1={merged.f1:.1%}")
ensemble.min_votes = 1  # reset


  min_votes=1:  P=39.9%  R=87.3%  F1=54.8%
  min_votes=2:  P=53.4%  R=84.4%  F1=65.5%
  min_votes=3:  P=90.1%  R=73.7%  F1=81.1%


## What to look for

- **Overall F1 vs LegalBERT alone (Notebook 01).** The ensemble should win on most labels. If it doesn't, the predictors are making correlated errors — diversify the member set (different transformer backbone, larger spaCy model, etc.) before going deeper on hyperparameters.
- **CODE recall.** With Presidio in the mix this should be high regardless of what LegalBERT and spaCy missed.
- **min_votes precision/recall curve.** The intersection regime (`min_votes=2`) usually trades 5–10 points of recall for 3–5 points of precision. If you're noise-sensitive, that might be the right operating point.

Notebook 03 plots all six models (Phase 1 spaCy, Phase 2 HF + Presidio + RoBERTa, Phase 6 LegalBERT + Ensemble) side by side.
